# Robust Sparse EM with Extended Kalman Filter (EKF)

## Non-Linear State-Space Model Implementation

This notebook implements the Robust Sparse EM model using **Extended Kalman Filter (EKF)** instead of the standard Kalman filter. EKF is used when the state transition or observation models are **non-linear**.

### Key Differences:
- **Standard Kalman Filter**: For linear models (z_t = A*z_{t-1} + w_t)
- **Extended Kalman Filter**: For non-linear models (z_t = f(z_{t-1}) + w_t)
- EKF uses Jacobian matrices to linearize the non-linear functions around the current state estimate

---

## 1. Setup and Imports

In [ ]:
import numpy as np
import scanpy as sc
import pandas as pd
from scipy.linalg import inv
from scipy.stats import multivariate_normal
from sklearn.decomposition import PCA
from sklearn.covariance import MinCovDet
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)

print("Libraries imported successfully!")

## 2. Extended Kalman Filter Class

In [ ]:
class RobustSparseStateSpaceEM_EKF:
    """
    Robust Sparse EM with Extended Kalman Filter for Non-Linear State-Space Models
    
    Features:
    - VAR[p] embedding for time-delayed effects
    - Extended Kalman Filter for non-linear dynamics
    - Robust outlier detection (detect-and-reject)
    - Sparse regularization (graphical lasso)
    - Constrained optimization (preserve VAR structure)
    
    Non-linear functions:
    - f(z): Non-linear state transition function
    - h(z): Non-linear observation function
    - J_f(z): Jacobian of state transition
    - J_h(z): Jacobian of observation function
    """
    
    def __init__(self, n_states, n_features, n_time_points, var_order=2,
                 l1_lambda=0.1, outlier_threshold=2.0, initialization='pca',
                 reg_strength=1e-6, nonlinearity='tanh'):
        """
        Parameters:
        -----------
        n_states : int - Number of latent states (K)
        n_features : int - PCA dimensions (D)
        n_time_points : int - Number of time points (T)
        var_order : int - VAR order (p) for time delays
        l1_lambda : float - L1 regularization strength for sparsity
        outlier_threshold : float - Threshold for outlier detection (Mahalanobis)
        initialization : str - 'pca', 'random', or 'clusters'
        reg_strength : float - Regularization for numerical stability
        nonlinearity : str - Type of non-linearity ('tanh', 'sigmoid', 'relu', 'polynomial')
        """
        self.n_states = n_states
        self.n_features = n_features
        self.n_time_points = n_time_points
        self.var_order = var_order
        self.l1_lambda = l1_lambda
        self.outlier_threshold = outlier_threshold
        self.reg_strength = reg_strength
        self.initialization = initialization
        self.nonlinearity = nonlinearity
        
        # Augmented dimensions
        self.augmented_states = n_states * var_order
        
        # Parameters
        self.A_matrices = None  # List of VAR matrices [A^(1), ..., A^(p)]
        self.A_aug = None  # Augmented transition matrix
        self.C = None  # Observation matrix (D × K*p)
        self.Q = None  # Process noise (K × K)
        self.Q_aug = None  # Augmented process noise
        self.R = None  # Observation noise (D × D)
        self.mu0 = None  # Initial state mean (K*p)
        self.Sigma0 = None  # Initial state covariance (K*p × K*p)
        
        # Storage
        self.X_smooth = None
        self.P_smooth = None
        self.outlier_flags = None
        self.log_likelihood = []
    
    def _nonlinear_transition(self, z):
        """
        Non-linear state transition function f(z)
        
        Parameters:
        -----------
        z : array - State vector
        
        Returns:
        --------
        f(z) : array - Transformed state
        """
        if self.nonlinearity == 'tanh':
            return np.tanh(z)
        elif self.nonlinearity == 'sigmoid':
            return 1 / (1 + np.exp(-z))
        elif self.nonlinearity == 'relu':
            return np.maximum(0, z)
        elif self.nonlinearity == 'polynomial':
            return z + 0.1 * z**2  # Quadratic non-linearity
        else:
            return z  # Linear (fallback)
    
    def _jacobian_transition(self, z):
        """
        Jacobian of non-linear transition function J_f(z)
        
        Parameters:
        -----------
        z : array - State vector
        
        Returns:
        --------
        J_f : array - Jacobian matrix
        """
        if self.nonlinearity == 'tanh':
            return np.diag(1 - np.tanh(z)**2)
        elif self.nonlinearity == 'sigmoid':
            sig = 1 / (1 + np.exp(-z))
            return np.diag(sig * (1 - sig))
        elif self.nonlinearity == 'relu':
            return np.diag((z > 0).astype(float))
        elif self.nonlinearity == 'polynomial':
            return np.diag(1 + 0.2 * z)
        else:
            return np.eye(len(z))  # Linear (identity)
    
    def _nonlinear_observation(self, z):
        """
        Non-linear observation function h(z)
        
        Parameters:
        -----------
        z : array - State vector
        
        Returns:
        --------
        h(z) : array - Transformed observation
        """
        # Apply non-linearity to current state only
        z_current = z[:self.n_states]
        z_nonlinear = self._nonlinear_transition(z_current)
        
        # Combine with linear observation
        result = np.zeros_like(z)
        result[:self.n_states] = z_nonlinear
        result[self.n_states:] = z[self.n_states:]  # Past states unchanged
        
        return result
    
    def _jacobian_observation(self, z):
        """
        Jacobian of non-linear observation function J_h(z)
        
        Parameters:
        -----------
        z : array - State vector
        
        Returns:
        --------
        J_h : array - Jacobian matrix
        """
        J = np.zeros((self.augmented_states, self.augmented_states))
        
        # Jacobian for current state (non-linear)
        z_current = z[:self.n_states]
        J_current = self._jacobian_transition(z_current)
        J[:self.n_states, :self.n_states] = J_current
        
        # Past states (identity)
        J[self.n_states:, self.n_states:] = np.eye(self.augmented_states - self.n_states)
        
        return J
    
    def initialize_parameters(self, Y_by_time):
        """Initialize parameters with VAR structure"""
        all_data = np.vstack([y for y in Y_by_time if y is not None])
        
        if self.initialization == 'pca':
            print("Initializing with PCA and VAR structure (EKF version)...")
            
            # PCA for C
            pca = PCA(n_components=self.n_states)
            pca.fit(all_data)
            self.C = pca.components_.T
            
            # Initialize C_aug
            self.C_aug = np.zeros((self.n_features, self.augmented_states))
            self.C_aug[:, :self.n_states] = self.C
            
            # Initialize states
            self.X_smooth = []
            for y in Y_by_time:
                if y is not None:
                    z = pca.transform(y)
                    X = np.zeros((y.shape[0], self.augmented_states))
                    X[:, :self.n_states] = z
                    self.X_smooth.append(X)
                else:
                    self.X_smooth.append(np.zeros((1, self.augmented_states)))
            
            # Initialize A matrices
            self.A_matrices = []
            for k in range(1, self.var_order + 1):
                A_k = np.random.randn(self.n_states, self.n_states) * 0.01
                self.A_matrices.append(A_k)
            
            self._assemble_augmented_A()
            
            # Initialize Q
            self.Q = self.reg_strength * np.eye(self.n_states)
            self.Q_aug = np.zeros((self.augmented_states, self.augmented_states))
            self.Q_aug[:self.n_states, :self.n_states] = self.Q
            
            # Initialize R
            residuals = []
            for y, X in zip(Y_by_time, self.X_smooth):
                if y is not None:
                    recon = X @ self.C_aug.T
                    residuals.append(y - recon)
            if residuals:
                all_residuals = np.vstack(residuals)
                self.R = np.cov(all_residuals.T) + self.reg_strength * np.eye(self.n_features)
            else:
                self.R = self.reg_strength * np.eye(self.n_features)
            
            self.mu0 = np.zeros(self.augmented_states)
            self.Sigma0 = self.reg_strength * np.eye(self.augmented_states)
        
        print("Initialization complete")
    
    def _assemble_augmented_A(self):
        """Assemble augmented transition matrix from VAR matrices"""
        self.A_aug = np.zeros((self.augmented_states, self.augmented_states))
        
        top_row_start = 0
        for k, A_k in enumerate(self.A_matrices):
            self.A_aug[:self.n_states, top_row_start:top_row_start+self.n_states] = A_k
            top_row_start += self.n_states
        
        for i in range(1, self.var_order):
            row_start = i * self.n_states
            col_start = (i - 1) * self.n_states
            self.A_aug[row_start:row_start+self.n_states, 
                      col_start:col_start+self.n_states] = np.eye(self.n_states)
    
    def _detect_outlier(self, y_t, z_pred, P_pred):
        """Detect outlier using Mahalanobis distance"""
        y_pred = self.C @ z_pred
        S = self.C @ P_pred @ self.C.T + self.R
        
        diff = y_t - y_pred
        try:
            mahalanobis_dist = np.sqrt(diff @ inv(S) @ diff.T)
        except:
            mahalanobis_dist = np.inf
        
        is_outlier = mahalanobis_dist > self.outlier_threshold
        
        return is_outlier, mahalanobis_dist
    
    def extended_kalman_filter_robust(self, Y_by_time):
        """
        Extended Kalman Filter with detect-and-reject
        
        Returns:
        --------
        X_filt : list of arrays - Filtered state estimates
        P_filt : list of arrays - Filtered covariances
        X_pred : list of arrays - Predicted state estimates
        P_pred : list of arrays - Predicted covariances
        outlier_flags : list of arrays - Outlier detection flags
        """
        X_filt = []
        P_filt = []
        X_pred = []
        P_pred = []
        outlier_flags = []
        
        X_pred.append(self.mu0)
        P_pred.append(self.Sigma0)
        
        for t in range(self.n_time_points):
            if Y_by_time[t] is None:
                X_filt.append(X_pred[t])
                P_filt.append(P_pred[t])
                outlier_flags.append(np.array([False]))
            else:
                n_cells = Y_by_time[t].shape[0]
                X_filt_t = []
                P_filt_t = []
                outlier_flags_t = []
                
                for i in range(n_cells):
                    y_t = Y_by_time[t][i]
                    z_pred = X_pred[t][:self.n_states]
                    P_pred_current = P_pred[t][:self.n_states, :self.n_states]
                    
                    # Detect outlier
                    is_outlier, mahalanobis_dist = self._detect_outlier(
                        y_t, z_pred, P_pred_current
                    )
                    outlier_flags_t.append(is_outlier)
                    
                    if is_outlier:
                        R_adaptive = np.eye(self.n_features) * 1e10
                    else:
                        R_adaptive = self.R
                    
                    # EKF Prediction Step (non-linear)
                    # z_pred = f(A_aug * z_prev)
                    z_pred_nonlinear = self._nonlinear_transition(self.A_aug @ X_pred[t])
                    z_pred_nonlinear = z_pred_nonlinear[:self.n_states]
                    
                    # Jacobian for linearization
                    J_f = self._jacobian_transition(self.A_aug @ X_pred[t])
                    J_f = J_f[:self.n_states, :self.n_states]
                    
                    # Predict covariance
                    P_pred_nonlinear = J_f @ P_pred_current @ J_f.T + self.Q
                    
                    # EKF Update Step
                    # Jacobian of observation
                    J_h = self._jacobian_observation(z_pred_nonlinear)
                    J_h = J_h[:self.n_states, :self.n_states]
                    
                    # Innovation covariance
                    S = self.C @ J_h @ P_pred_nonlinear @ J_h.T @ self.C.T + R_adaptive
                    S = S + self.reg_strength * np.eye(self.n_features)
                    
                    # Kalman gain
                    K = P_pred_nonlinear @ J_h.T @ self.C.T @ inv(S)
                    
                    # Update
                    innovation = y_t - self.C @ z_pred_nonlinear
                    z_filt = z_pred_nonlinear + K @ innovation
                    P_filt_current = (np.eye(self.n_states) - K @ self.C @ J_h) @ P_pred_nonlinear
                    
                    # Augment filtered state
                    X_filt_i = np.zeros(self.augmented_states)
                    X_filt_i[:self.n_states] = z_filt
                    if t > 0 and isinstance(X_filt[t-1], np.ndarray):
                        if X_filt[t-1].ndim == 1:
                            X_filt_i[self.n_states:] = X_filt[t-1][:self.n_states*(self.var_order-1)]
                        else:
                            X_filt_i[self.n_states:] = X_filt[t-1][0, :self.n_states*(self.var_order-1)]
                    
                    X_filt_t.append(X_filt_i)
                    P_filt_t.append(P_filt_current)
                
                X_filt.append(np.array(X_filt_t))
                P_filt.append(np.array(P_filt_t))
                outlier_flags.append(np.array(outlier_flags_t))
            
            # Predict next
            if t < self.n_time_points - 1:
                if isinstance(X_filt[t], np.ndarray) and X_filt[t].ndim > 1:
                    X_pred_t = (self.A_aug @ X_filt[t].T).T
                    P_pred_t = []
                    for i in range(len(P_filt[t])):
                        P_aug = np.zeros((self.augmented_states, self.augmented_states))
                        P_aug[:self.n_states, :self.n_states] = P_filt[t][i]
                        P_pred_i = self.A_aug @ P_aug @ self.A_aug.T + self.Q_aug
                        P_pred_t.append(P_pred_i)
                    P_pred_t = np.array(P_pred_t)
                else:
                    X_pred_t = self.A_aug @ X_filt[t]
                    P_aug = np.zeros((self.augmented_states, self.augmented_states))
                    P_aug[:self.n_states, :self.n_states] = P_filt[t]
                    P_pred_t = self.A_aug @ P_aug @ self.A_aug.T + self.Q_aug
                
                X_pred.append(X_pred_t)
                P_pred.append(P_pred_t)
        
        return X_filt, P_filt, X_pred, P_pred, outlier_flags
    
    def rts_smoother_augmented(self, X_filt, P_filt, X_pred, P_pred):
        """RTS Smoother for augmented states"""
        n_time = len(X_filt)
        X_smooth = [None] * n_time
        P_smooth = [None] * n_time
        
        X_smooth[-1] = X_filt[-1]
        P_smooth[-1] = P_filt[-1]
        
        for t in range(n_time - 2, -1, -1):
            if isinstance(X_filt[t], np.ndarray) and X_filt[t].ndim > 1:
                n_cells = X_filt[t].shape[0]
                X_smooth_t = []
                P_smooth_t = []
                
                for i in range(n_cells):
                    P_pred_next = P_pred[t+1] if isinstance(P_pred[t+1], np.ndarray) and P_pred[t+1].ndim == 2 else P_pred[t+1][i]
                    J = P_filt[t][i] @ self.A_aug.T @ inv(P_pred_next + self.reg_strength * np.eye(self.augmented_states))
                    X_smooth_i = X_filt[t][i] + J @ (X_smooth[t+1][i] - self.A_aug @ X_filt[t][i])
                    P_smooth_i = P_filt[t][i] + J @ (P_smooth[t+1][i] - P_pred_next) @ J.T
                    X_smooth_t.append(X_smooth_i)
                    P_smooth_t.append(P_smooth_i)
                
                X_smooth[t] = np.array(X_smooth_t)
                P_smooth[t] = np.array(P_smooth_t)
            else:
                P_pred_next = P_pred[t+1] if isinstance(P_pred[t+1], np.ndarray) and P_pred[t+1].ndim == 2 else P_pred[t+1][0]
                J = P_filt[t] @ self.A_aug.T @ inv(P_pred_next + self.reg_strength * np.eye(self.augmented_states))
                X_smooth[t] = X_filt[t] + J @ (X_smooth[t+1] - self.A_aug @ X_filt[t])
                P_smooth[t] = P_filt[t] + J @ (P_smooth[t+1] - P_pred_next) @ J.T
        
        return X_smooth, P_smooth
    
    def compute_sufficient_statistics_augmented(self, Y_by_time, X_smooth, P_smooth):
        """Compute sufficient statistics for augmented model"""
        E_XzT = [np.zeros((self.n_states, self.n_states)) for _ in range(self.var_order)]
        E_zzT_prev = np.zeros((self.n_states, self.n_states))
        E_yzT = np.zeros((self.n_features, self.n_states))
        E_zzT_obs = np.zeros((self.n_states, self.n_states))
        E_yyT = np.zeros((self.n_features, self.n_features))
        
        n_pairs = 0
        n_obs = 0
        
        for t in range(self.n_time_points):
            if X_smooth[t] is not None:
                for i in range(X_smooth[t].shape[0]):
                    z_t = X_smooth[t][i, :self.n_states]
                    P_t = P_smooth[t][i][:self.n_states, :self.n_states]
                    
                    E_zzT_obs += P_t + np.outer(z_t, z_t)
                    n_obs += 1
                
                if Y_by_time[t] is not None:
                    for i in range(Y_by_time[t].shape[0]):
                        z_t = X_smooth[t][i, :self.n_states]
                        E_yzT += np.outer(Y_by_time[t][i], z_t)
                        E_yyT += np.outer(Y_by_time[t][i], Y_by_time[t][i])
                
                if t > 0 and X_smooth[t-1] is not None:
                    n_cells_t = X_smooth[t].shape[0]
                    n_cells_t1 = X_smooth[t-1].shape[0]
                    n_pairs_cell = min(n_cells_t, n_cells_t1)
                    
                    for i in range(n_pairs_cell):
                        z_t = X_smooth[t][i, :self.n_states]
                        
                        for k in range(self.var_order):
                            if t - k >= 0:
                                lag_idx = min(i, X_smooth[t-k].shape[0]-1)
                                z_t_k = X_smooth[t-k][lag_idx, :self.n_states]
                                E_XzT[k] += np.outer(z_t, z_t_k)
                        
                        E_zzT_prev += np.outer(z_t, z_t)
                        n_pairs += 1
        
        if n_pairs > 0:
            for k in range(self.var_order):
                E_XzT[k] /= n_pairs
            E_zzT_prev /= n_pairs
        if n_obs > 0:
            E_zzT_obs /= n_obs
            E_yzT /= n_obs
            E_yyT /= n_obs
        
        return E_XzT, E_zzT_prev, E_yzT, E_zzT_obs, E_yyT
    
    def soft_threshold(self, x, lambda_val):
        """Soft thresholding operator for Lasso"""
        return np.sign(x) * np.maximum(np.abs(x) - lambda_val, 0)
    
    def graphical_lasso_update(self, E_XzT, E_zzT_prev):
        """Update VAR matrices using graphical lasso"""
        for k in range(self.var_order):
            for i in range(self.n_states):
                for j in range(self.n_states):
                    numerator = self.soft_threshold(E_XzT[k][i, j], self.l1_lambda)
                    denominator = E_zzT_prev[j, j] + self.l1_lambda
                    self.A_matrices[k][i, j] = numerator / denominator
        
        self._assemble_augmented_A()
    
    def m_step_sparse(self, E_XzT, E_zzT_prev, E_yzT, E_zzT_obs, E_yyT):
        """M-Step with sparse regularization"""
        self.graphical_lasso_update(E_XzT, E_zzT_prev)
        
        try:
            self.C = E_yzT @ inv(E_zzT_obs + self.reg_strength * np.eye(self.n_states))
        except np.linalg.LinAlgError:
            self.C = E_yzT @ np.linalg.pinv(E_zzT_obs + self.reg_strength * np.eye(self.n_states))
        
        self.C_aug = np.zeros((self.n_features, self.augmented_states))
        self.C_aug[:, :self.n_states] = self.C
        
        self.Q = (E_zzT_obs - self.A_matrices[0] @ E_XzT[0].T - 
                  E_XzT[0] @ self.A_matrices[0].T + 
                  self.A_matrices[0] @ E_zzT_prev @ self.A_matrices[0].T)
        self.Q = (self.Q + self.Q.T) / 2
        self.Q = np.maximum(self.Q, self.reg_strength)
        
        self.Q_aug = np.zeros((self.augmented_states, self.augmented_states))
        self.Q_aug[:self.n_states, :self.n_states] = self.Q
        
        self.R = (E_yyT - self.C @ E_yzT.T - E_yzT @ self.C.T + 
                  self.C @ E_zzT_obs @ self.C.T)
        self.R = (self.R + self.R.T) / 2
        self.R = np.maximum(self.R, self.reg_strength)
        
        if self.X_smooth[0] is not None:
            self.mu0 = self.X_smooth[0].mean(axis=0)
            self.Sigma0 = np.cov(self.X_smooth[0].T) + self.reg_strength * np.eye(self.augmented_states)
    
    def fit(self, Y_by_time, max_iter=100, tol=1e-6, verbose=True):
        """Run robust sparse EM with EKF"""
        self.initialize_parameters(Y_by_time)
        prev_ll = -np.inf
        
        for iteration in range(max_iter):
            # E-step with EKF
            X_filt, P_filt, X_pred, P_pred, outlier_flags = self.extended_kalman_filter_robust(Y_by_time)
            X_smooth, P_smooth = self.rts_smoother_augmented(X_filt, P_filt, X_pred, P_pred)
            
            # Compute sufficient statistics
            E_XzT, E_zzT_prev, E_yzT, E_zzT_obs, E_yyT = \
                self.compute_sufficient_statistics_augmented(Y_by_time, X_smooth, P_smooth)
            
            # M-step
            self.m_step_sparse(E_XzT, E_zzT_prev, E_yzT, E_zzT_obs, E_yyT)
            
            self.X_smooth = X_smooth
            self.P_smooth = P_smooth
            self.outlier_flags = outlier_flags
            
            current_ll = -sum([np.sum(flags) for flags in outlier_flags])
            self.log_likelihood.append(current_ll)
            
            if verbose:
                n_outliers = sum([np.sum(flags) for flags in outlier_flags])
                sparsity = np.mean([np.sum(np.abs(A_k) < 1e-3) for A_k in self.A_matrices])
                print(f"Iteration {iteration + 1}: Outliers={n_outliers}, Sparsity={sparsity:.2%}")
            
            if abs(current_ll - prev_ll) < tol:
                if verbose:
                    print(f"Converged at iteration {iteration + 1}")
                break
            
            prev_ll = current_ll
        
        return self

print("RobustSparseStateSpaceEM_EKF class defined successfully!")

## 3. Comparison: Standard KF vs EKF

In [ ]:
# Visualize non-linear functions
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

x = np.linspace(-3, 3, 100)

# Tanh
axes[0, 0].plot(x, np.tanh(x), 'b-', linewidth=2)
axes[0, 0].set_title('Tanh Non-linearity')
axes[0, 0].set_xlabel('x')
axes[0, 0].set_ylabel('tanh(x)')
axes[0, 0].grid(True, alpha=0.3)

# Sigmoid
axes[0, 1].plot(x, 1/(1+np.exp(-x)), 'r-', linewidth=2)
axes[0, 1].set_title('Sigmoid Non-linearity')
axes[0, 1].set_xlabel('x')
axes[0, 1].set_ylabel('sigmoid(x)')
axes[0, 1].grid(True, alpha=0.3)

# ReLU
axes[1, 0].plot(x, np.maximum(0, x), 'g-', linewidth=2)
axes[1, 0].set_title('ReLU Non-linearity')
axes[1, 0].set_xlabel('x')
axes[1, 0].set_ylabel('relu(x)')
axes[1, 0].grid(True, alpha=0.3)

# Polynomial
axes[1, 1].plot(x, x + 0.1*x**2, 'm-', linewidth=2)
axes[1, 1].set_title('Polynomial Non-linearity')
axes[1, 1].set_xlabel('x')
axes[1, 1].set_ylabel('x + 0.1*x²')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Non-linear functions available for EKF:")
print("1. Tanh - Smooth, bounded (-1 to 1)")
print("2. Sigmoid - Smooth, bounded (0 to 1)")
print("3. ReLU - Piecewise linear, unbounded")
print("4. Polynomial - Quadratic, unbounded")

## 4. Key Differences: KF vs EKF

In [ ]:
print("=== Key Differences: Standard KF vs Extended KF ===")
print()
print("Standard Kalman Filter:")
print("  - Model: Linear (z_t = A*z_{t-1} + w_t)")
print("  - Prediction: z_pred = A * z_prev")
print("  - Covariance: P_pred = A * P_prev * A^T + Q")
print("  - Exact solution")
print("  - Faster computation")
print("  - Optimal for linear systems")
print()
print("Extended Kalman Filter:")
print("  - Model: Non-linear (z_t = f(z_{t-1}) + w_t)")
print("  - Prediction: z_pred = f(A * z_prev)")
print("  - Covariance: P_pred = J_f * P_prev * J_f^T + Q")
print("  - Approximate solution (linearization)")
print("  - Slower computation (Jacobian calculation)")
print("  - Suboptimal but works for non-linear systems")
print()
print("When to use EKF:")
print("  - Non-linear state transitions")
print("  - Non-linear observation models")
print("  - Saturating dynamics (e.g., gene expression saturation)")
print("  - Threshold effects")
print()
print("When to use Standard KF:")
print("  - Linear dynamics")
print("  - Gaussian noise assumptions")
print("  - Computational efficiency priority")
print("  - Your current data (PCA space is approximately linear)")

## 5. Example Usage (requires preprocessed data)

In [ ]:
# Example: Initialize EKF model
# Note: This requires Y_by_time from preprocessing (see main notebook)

print("=== EKF Model Initialization ===")
print()
print("To use the EKF model:")
print("1. Run preprocessing from main notebook to get Y_by_time")
print("2. Initialize model with nonlinearity parameter:")
print()
print("model_ekf = RobustSparseStateSpaceEM_EKF(")
print("    n_states=10,")
print("    n_features=n_pcs,")
print("    n_time_points=len(Y_by_time),")
print("    var_order=2,")
print("    l1_lambda=0.1,")
print("    outlier_threshold=2.0,")
print("    initialization='pca',")
print("    reg_strength=1e-6,")
print("    nonlinearity='tanh'  # Options: 'tanh', 'sigmoid', 'relu', 'polynomial'")
print(")")
print()
print("model_ekf = model_ekf.fit(Y_by_time, max_iter=100, verbose=True)")
print()
print("=== Comparison with Standard KF ===")
print()
print("Advantages of EKF:")
print("  - Can model non-linear biological dynamics")
print("  - Captures saturation effects (e.g., gene expression limits)")
print("  - More flexible for complex regulatory networks")
print()
print("Disadvantages of EKF:")
print("  - Slower (Jacobian calculations)")
print("  - Approximate (linearization error)")
print("  - May diverge if non-linearity is strong")
print("  - More hyperparameters (nonlinearity type)")
print()
print("Recommendation for your data:")
print("  - Use Standard KF (from main notebook) as baseline")
print("  - Try EKF with 'tanh' if you suspect non-linear dynamics")
print("  - Compare results: sparsity, reconstruction error, biological interpretability")
print("  - If EKF doesn't improve results, stick with Standard KF")

## 6. Summary

In [ ]:
print("=== Extended Kalman Filter Implementation Summary ===")
print()
print("This notebook provides:")
print("1. RobustSparseStateSpaceEM_EKF class with non-linear dynamics")
print("2. Multiple non-linearity options (tanh, sigmoid, relu, polynomial)")
print("3. Jacobian calculation for linearization")
print("4. Robust outlier detection (same as standard KF)")
print("5. Graphical lasso for sparse regularization (same as standard KF)")
print()
print("Key Implementation Details:")
print("- Non-linear transition: f(z) = tanh(A*z) or other")
print("- Jacobian: J_f(z) = diag(1 - tanh²(A*z))")
print("- Linearization at each time step")
print("- Same VAR[p] structure as standard KF")
print("- Same robust detection and sparse regularization")
print()
print("Use Cases:")
print("- If standard KF shows poor convergence")
print("- If you suspect non-linear biological dynamics")
print("- If reconstruction error is high with linear model")
print("- For comparison and validation of linear assumption")